## BLOQUE 1 — Instalar dependencias

In [1]:
!pip install -q pymupdf sentence-transformers chromadb transformers accelerate
print("Todo instalado")

Todo instalado


In [2]:
from huggingface_hub import login
login("token")

## BLOQUE 2 — Cargar modelo generador desde HuggingFace


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-2-2b-it"

print(f"Descargando {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
print("Gemma cargado")

Descargando google/gemma-2-2b-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma cargado


##  BLOQUE 3 — Subir los PDFs del corpus


In [4]:
from google.colab import files
import os

print(" Sube tus PDFs del corpus:")
uploaded = files.upload()

os.makedirs("corpus_pdfs", exist_ok=True)
pdf_paths = []
for nombre, contenido in uploaded.items():
    ruta = f"corpus_pdfs/{nombre}"
    with open(ruta, "wb") as f:
        f.write(contenido)
    pdf_paths.append(ruta)
    print(f"   {nombre} ({len(contenido)//1024} KB)")

print(f"\n {len(pdf_paths)} PDF(s) cargados")

 Sube tus PDFs del corpus:


Saving Actor-Profile-The-Jalisco-New-Generation-Cartel-CJNG-Spanish-Translation.pdf to Actor-Profile-The-Jalisco-New-Generation-Cartel-CJNG-Spanish-Translation (1).pdf
Saving Desplazamiento_Forzado_Interno.pdf to Desplazamiento_Forzado_Interno (1).pdf
Saving DesplazamientoForzado.pdf to DesplazamientoForzado (1).pdf
Saving extorsiones.pdf to extorsiones (1).pdf
Saving HALLAZGOS2023.pdf to HALLAZGOS2023 (1).pdf
Saving Medicion-multidimensional-de-la-pobreza-en-Mexico.pdf to Medicion-multidimensional-de-la-pobreza-en-Mexico (1).pdf
Saving Militarizacion.pdf to Militarizacion (1).pdf
Saving ong.pdf to ong (1).pdf
Saving pregunta1.pdf to pregunta1 (1).pdf
Saving tierra_caliente_narco_autodefensa.pdf to tierra_caliente_narco_autodefensa (1).pdf
Saving violencia rural.pdf to violencia rural (1).pdf
Saving violencia urbana.pdf to violencia urbana (1).pdf
   Actor-Profile-The-Jalisco-New-Generation-Cartel-CJNG-Spanish-Translation (1).pdf (1273 KB)
   Desplazamiento_Forzado_Interno (1).pdf (103

## BLOQUE 4 — Extraer texto de PDFs y hacer chunking

In [5]:
import fitz  # PyMuPDF
import re
import os

def extraer_texto_pdf(ruta):

    doc = fitz.open(ruta)

    metadata = doc.metadata

    info = {
        "titulo": metadata.get("title", ""),
        "autor": metadata.get("author", ""),
        "subject": metadata.get("subject", ""),
        "creator": metadata.get("creator", "")
    }

    paginas = []

    for num, pag in enumerate(doc, 1):

        texto = pag.get_text("text").strip()

        if texto:
            paginas.append({
                "pagina": num,
                "texto": texto
            })

    doc.close()

    return paginas, info


def limpiar(texto):

    texto = re.sub(r'\n{3,}', '\n\n', texto)
    texto = re.sub(r' {2,}', ' ', texto)

    return texto.strip()


def chunking(texto, tam=350, solape=70):

    palabras = texto.split()

    chunks = []

    i = 0

    while i < len(palabras):

        chunk = ' '.join(palabras[i:i+tam])

        if len(chunk) > 60:
            chunks.append(chunk)

        i += tam - solape

    return chunks


# ── Procesar PDFs ──────────────────────────────────────────

print("Extrayendo texto.\n")

documentos_chunks = []

for ruta in pdf_paths:

    nombre = os.path.basename(ruta).replace(".pdf", "")

    paginas, info = extraer_texto_pdf(ruta)

    if not paginas:
        print(f"   {nombre}: sin texto extraíble")
        continue

    texto_total = "\n\n".join(
        [limpiar(p["texto"]) for p in paginas]
    )

    chunks = chunking(texto_total)

    for i, chunk in enumerate(chunks):

        documentos_chunks.append({
            "id": f"{nombre}_c{i:03d}",
            "fuente": nombre,
            "autor": info.get("autor", ""),
            "titulo": info.get("titulo", ""),
            "subject": info.get("subject", ""),
            "creator": info.get("creator", ""),
            "pag": f"aprox. pág {int(i*0.8)+1}",
            "texto": chunk
        })

    print(
        f"   {nombre}: {len(paginas)} págs → {len(chunks)} chunks"
    )

print(
    f"\nTotal: {len(documentos_chunks)} chunks de {len(pdf_paths)} PDFs"
)

Extrayendo texto.

   Actor-Profile-The-Jalisco-New-Generation-Cartel-CJNG-Spanish-Translation (1): 14 págs → 20 chunks
   Desplazamiento_Forzado_Interno (1): 96 págs → 139 chunks
   DesplazamientoForzado (1): 38 págs → 33 chunks
   extorsiones (1): 60 págs → 38 chunks
   HALLAZGOS2023 (1): 207 págs → 425 chunks
   Medicion-multidimensional-de-la-pobreza-en-Mexico (1): 14 págs → 11 chunks
   Militarizacion (1): 30 págs → 32 chunks
   ong (1): 20 págs → 36 chunks
   pregunta1 (1): 86 págs → 13 chunks
   tierra_caliente_narco_autodefensa (1): 15 págs → 31 chunks
   violencia rural (1): 64 págs → 62 chunks
   violencia urbana (1): 189 págs → 237 chunks

Total: 1077 chunks de 12 PDFs


##  BLOQUE 5 — Embeddings + ChromaDB (Vector Store)

In [6]:
from sentence_transformers import SentenceTransformer
import chromadb, time

print(" Cargando modelo de embeddings...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(" Embeddings listos")

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("corpus_mx")
except:
    pass
coleccion = chroma_client.create_collection(
    name="corpus_mx",
    metadata={"hnsw:space": "cosine"}
)

print(f" Indexando {len(documentos_chunks)} chunks...")
t0 = time.time()
BATCH = 32
for i in range(0, len(documentos_chunks), BATCH):
    batch = documentos_chunks[i:i+BATCH]
    embs = embed_model.encode([d["texto"] for d in batch], show_progress_bar=False)
    coleccion.add(
        ids=[d["id"] for d in batch],
        embeddings=embs.tolist(),
        documents=[d["texto"] for d in batch],
        metadatas=[{"fuente": d["fuente"], "pag": d["pag"]} for d in batch]
    )
    print(f"  {min(i+BATCH, len(documentos_chunks))}/{len(documentos_chunks)}", end="\r")

print(f"\n Vector store listo en {time.time()-t0:.1f}s | Docs: {coleccion.count()}")

 Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embeddings listos
 Indexando 1077 chunks...
  1077/1077
 Vector store listo en 8.1s | Docs: 1077


## BLOQUE 6 — Función de Recuperación RAG (Top-K)

In [7]:
def recuperar_chunks(consulta, k=3):
    t0 = time.time()
    q_emb = embed_model.encode([consulta])
    res = coleccion.query(query_embeddings=q_emb.tolist(), n_results=k)
    lat = round((time.time()-t0)*1000, 1)
    chunks = [
        {
            "id":       res["ids"][0][i],
            "texto":    res["documents"][0][i],
            "fuente":   res["metadatas"][0][i]["fuente"],
            "pag":      res["metadatas"][0][i]["pag"],
            "dist":     round(res["distances"][0][i], 4)
        }
        for i in range(len(res["ids"][0]))
    ]
    return chunks, lat

def mostrar_chunks(chunks, lat):
    print(f"    RAG: {lat} ms | {len(chunks)} chunks recuperados:")
    for i, c in enumerate(chunks, 1):
        rel = round((1 - c["dist"]) * 100, 1)
        print(f"  ┌─ [{i}] {c['id']}")
        print(f"  │  Fuente: {c['fuente']} | {c['pag']} | Relevancia: {rel}%")
        print(f"  └─ {c['texto'][:130].strip()}...")
        print()

# Prueba
chunks_test, lat_test = recuperar_chunks("violencia homicidios México")
mostrar_chunks(chunks_test, lat_test)

    RAG: 132.2 ms | 3 chunks recuperados:
  ┌─ [1] pregunta1 (1)_c012
  │  Fuente: pregunta1 (1) | aprox. pág 10 | Relevancia: 76.8%
  └─ 15 de julio 84 Casos de violencia en los últimos días en México Tulancingo, Hidalgo Cuatro personas fueron baleadas la madrugada d...

  ┌─ [2] pregunta1 (1)_c010
  │  Fuente: pregunta1 (1) | aprox. pág 9 | Relevancia: 74.9%
  └─ Caliente. Ataques que provocaron lesiones a un joven y daños materiales en algunas viviendas. 8 y 9 de julio 76 Casos de violencia...

  ┌─ [3] Militarizacion (1)_c010
  │  Fuente: Militarizacion (1) | aprox. pág 9 | Relevancia: 70.2%
  └─ crímenes de lesa humani- dad en México (informes presentados a la Corte Penal Internacio- nal), CMDPDH (2019). 3 Gobierno de Méxic...



## BLOQUE 7 — Dataset Fine-Tuning (entregable JSONL)


In [8]:
import json

DATASET_FT = [
    {
        "instruction": "¿Cuáles son los estados con mayor índice de homicidios dolosos?",
        "input": "Contexto del corpus disponible.",
        "output": (
            "Con base en los documentos del corpus, los estados con mayor incidencia "
            "de homicidios dolosos son los que el SESNSP documenta en sus reportes. "
            "Cada cifra debe contrastarse con el documento fuente para verificar el periodo.\n"
            "[Fuente: citar nombre del PDF y página correspondiente]"
        )
    },
    {
        "instruction": "¿Es culpa del gobierno toda la violencia en México?",
        "input": "",
        "output": (
            "Desde una perspectiva académica la causalidad es multifactorial. "
            "El corpus documenta tanto factores estructurales (desigualdad, mercados ilegales) "
            "como institucionales (impunidad, corrupción). "
            "¿Desde qué dimensión quieres analizar el fenómeno primero?"
        )
    },
    {
        "instruction": "Explícame todo sobre la violencia en México.",
        "input": "",
        "output": (
            "Es un fenómeno complejo. Para guiar el análisis necesito saber:\n"
            "1. ¿Te interesa partir de datos cuantitativos o factores causales?\n"
            "2. ¿Tu enfoque es comparar estrategias o entender grupos delictivos?\n"
            "3. ¿Tienes alguna región o periodo de especial interés?"
        )
    },
    {
        "instruction": "¿Qué impacto tiene la violencia sobre la deserción escolar?",
        "input": "",
        "output": (
            "La información proporcionada en el corpus no detalla este aspecto.\n"
            "El corpus se centra en seguridad pública e incidencia delictiva, "
            "no en indicadores educativos. Para responder con rigor sería necesario "
            "incorporar fuentes como SEP o UNICEF."
        )
    },
    {
        "instruction": "¿Cuántos turistas afecta la inseguridad en México?",
        "input": "",
        "output": (
            "La información proporcionada en el corpus no detalla este aspecto.\n"
            "El corpus no incluye documentos sobre turismo. "
            "Proporcionar cifras sin respaldo documental comprometería la integridad del análisis."
        )
    },
    {
        "instruction": "¿Cuáles son las discrepancias entre ONGs y fuentes oficiales en el registro de víctimas?",
        "input": "Contexto del corpus disponible.",
        "output": (
            "El corpus identifica brechas sistemáticas entre cifras gubernamentales y organizaciones "
            "de la sociedad civil. Las discrepancias se atribuyen a incentivos institucionales "
            "para subregistrar, cambios en criterios de clasificación y presiones políticas.\n"
            "[Fuente: ver chunks recuperados por el RAG]"
        )
    },
    {
        "instruction": "¿Qué estrategia es mejor: militarización o prevención social?",
        "input": "Contexto del corpus disponible.",
        "output": (
            "Antes de concluir conviene explorar desde qué criterio evaluamos:\n"
            "- ¿Priorizas reducción inmediata de delitos visibles?\n"
            "- ¿Priorizas derechos humanos y largo plazo?\n"
            "El corpus registra argumentos para ambas posiciones. "
            "La elección del criterio determina la conclusión.\n"
            "[Fuente: ver chunks recuperados por el RAG]"
        )
    },
]

with open("dataset_finetuning.jsonl", "w", encoding="utf-8") as f:
    for item in DATASET_FT:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"dataset_finetuning.jsonl guardado — {len(DATASET_FT)} ejemplos")

dataset_finetuning.jsonl guardado — 7 ejemplos


## BLOQUE 7B — Mitigación de alucinaciones y citas automáticas

In [9]:

UMBRAL_RELEVANCIA = 0.65

def hay_contexto_suficiente(resultados):
    try:
        distancias = resultados.get("distances", [[]])[0]
        if len(distancias) == 0:
            return False
        return min(distancias) <= UMBRAL_RELEVANCIA
    except:
        return True

def generar_citas(resultados):
    citas = []
    for meta in resultados.get("metadatas", [[]])[0]:
        fuente = meta.get("fuente","Documento")
        pagina = meta.get("pagina","N/D")
        citas.append(f"[{fuente}, pág. {pagina}]")
    return "\n".join(sorted(set(citas)))


## BLOQUE 8 — Motor del Tutor: RAG → HuggingFace


In [10]:
import time
import json
import torch
import warnings

warnings.filterwarnings("ignore")

# ── CATÁLOGO DE METADATOS (se actualiza automáticamente) ──────
def generar_catalogo():
    fuentes = {}
    for d in documentos_chunks:
        f = d["fuente"]
        if f not in fuentes:
            fuentes[f] = 0
        fuentes[f] += 1
    lineas = []
    for i, (nombre, n_chunks) in enumerate(sorted(fuentes.items()), 1):
        lineas.append(f"{i}. \"{nombre}\" ({n_chunks} chunks)")
    return "\n".join(lineas)

SYSTEM_PROMPT = f"""Eres un Tutor Académico experto en seguridad pública y violencia en México.

CATÁLOGO DE ARCHIVOS DEL CORPUS ({len(pdf_paths)} PDFs cargados):
{generar_catalogo()}

REGLAS ESTRICTAS:
1. Si preguntan cuántos PDFs hay, qué archivos hay, autores o temas: responde usando el CATÁLOGO de arriba.
2. Para preguntas analíticas: responde ÚNICAMENTE con la INFORMACIÓN proporcionada.
3. Si la respuesta analítica no está en la INFORMACIÓN: responde EXACTAMENTE: 'La información proporcionada en el corpus no detalla este aspecto.'
4. Siempre coloca al final: [Fuente: Nombre del Documento, Pág: X]
5. Responde SIEMPRE en español."""

MAX_TOKENS_MODELO    = 2048
MAX_TOKENS_RESPUESTA = 250
MAX_TOKENS_PROMPT    = MAX_TOKENS_MODELO - MAX_TOKENS_RESPUESTA


def recortar_chunks(chunks, max_chars=3000):
    resultado, total = [], 0
    for c in chunks:
        espacio = max_chars - total
        if espacio <= 80: break
        c2 = dict(c)
        c2["texto"] = c["texto"][:espacio] + ("..." if len(c["texto"]) > espacio else "")
        resultado.append(c2)
        total += len(c2["texto"])
    return resultado


def construir_prompt_gemma(consulta, chunks):
    chunks_ok = recortar_chunks(chunks)
    if not chunks_ok:
        contexto = "No se recuperaron documentos relevantes."
    else:
        partes = []
        for i, c in enumerate(chunks_ok, 1):
            rel = round((1 - c.get("dist", 0.5)) * 100, 1)
            pag = c.get("pag", "N/A")
            partes.append(
                f"[Doc {i} | Fuente: {c['fuente']} | Pág: {pag} | Rel: {rel}%]\n{c['texto']}"
            )
        contexto = "\n\n---\n\n".join(partes)

    return (
        f"<start_of_turn>user\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"INFORMACIÓN:\n{contexto}\n\n"
        f"PREGUNTA: {consulta}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )


def tutor_responde_gemma(consulta, k=3, verbose=True):
    try:
        chunks, lat_rag = recuperar_chunks(consulta, k=k)
        prompt = construir_prompt_gemma(consulta, chunks)

        inputs = tokenizer(
            prompt, return_tensors="pt", add_special_tokens=False
        ).to("cuda")

        if inputs["input_ids"].shape[1] > MAX_TOKENS_PROMPT:
            inputs["input_ids"]      = inputs["input_ids"][:, :MAX_TOKENS_PROMPT]
            inputs["attention_mask"] = inputs["attention_mask"][:, :MAX_TOKENS_PROMPT]

        t0 = time.time()
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_TOKENS_RESPUESTA,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        lat_gen = round((time.time() - t0) * 1000, 1)

        longitud_prompt = inputs["input_ids"].shape[1]
        respuesta_tokens = outputs[0][longitud_prompt:]
        respuesta = tokenizer.decode(
            respuesta_tokens, skip_special_tokens=True
        ).strip()

        # Limpiar líneas con basura numérica
        lineas_limpias = []
        for linea in respuesta.split("\n"):
            tokens = linea.strip().split()
            if len(tokens) > 3:
                ratio_numeros = sum(1 for t in tokens if t.replace(".","").replace(",","").isdigit()) / len(tokens)
                if ratio_numeros > 0.6:
                    continue
            lineas_limpias.append(linea)
        respuesta = "\n".join(lineas_limpias).strip()

        if not respuesta:
            respuesta = "La información proporcionada en el corpus no detalla este aspecto."

        return respuesta, chunks, lat_rag, lat_gen

    except Exception as e:
        print(f" Error: {e}")
        return "[ERROR INTERNO]: Generación fallida.", [], 0, 0


# ── EVALUACIÓN COMPLETA ───────────────────────────────────────
BANCO = [
    {"id":"Q1",  "n":1, "k":5,
     "q":"¿Cuáles son las tres entidades federativas con mayor índice de homicidios dolosos según los datos más recientes del corpus?",
     "rag_query":"Colima Zacatecas Morelos homicidios dolosos tasa entidades mayor índice nacional"},
    {"id":"Q2",  "n":1, "k":5,
     "q":"¿Qué organizaciones, cárteles o grupos delictivos se mencionan operando en la región de Tierra Caliente?",
     "rag_query":"Caballeros Templarios Familia Michoacana CJNG Tierra Caliente Michoacán narcotráfico autodefensa"},
    {"id":"Q3",  "n":1, "k":5,
     "q":"¿Cuáles son las cifras oficiales sobre el desplazamiento forzado interno durante el último sexenio documentado?",
     "rag_query":"cifras número personas desplazamiento forzado interno sexenio miles datos estadísticas oficiales"},
    {"id":"Q4",  "n":2, "k":6,
     "q":"¿Cuáles son las principales causas socioeconómicas que los autores asocian al incremento de la violencia urbana?",
     "rag_query":"causas socioeconómicas violencia urbana pobreza desigualdad desempleo factores determinantes"},
    {"id":"Q5",  "n":2, "k":6,
     "q":"Contrasta las estrategias de seguridad pública del corpus: ¿qué diferencias hay entre militarización y prevención social?",
     "rag_query":"militarización fuerzas armadas Guardia Nacional prevención social delito estrategia seguridad diferencias"},
    {"id":"Q6",  "n":2, "k":6,
     "q":"¿Cómo ha evolucionado la tasa de extorsión (cobro de piso) y qué sectores económicos son los más afectados?",
     "rag_query":"extorsión cobro piso evolución tasa sectores económicos afectados comercio transporte renta extorsiva"},
    {"id":"Q7",  "n":2, "k":6,
     "q":"¿Existe diferencia documentada en los tipos de violencia entre zonas rurales y zonas metropolitanas?",
     "rag_query":"diferencia violencia rural metropolitana urbano dominio campo municipios tipos características delitos"},
    {"id":"Q8",  "n":3, "k":6,
     "q":"¿Cuáles son las contradicciones entre ONGs y fuentes gubernamentales en el registro de víctimas?",
     "rag_query":"contradicciones discrepancias ONGs gobierno cifras víctimas registro subregistro datos brecha"},
    {"id":"Q9",  "n":3, "k":3,
     "q":"¿Qué impacto tiene la violencia documentada sobre la tasa de deserción escolar en zonas de alto conflicto?",
     "rag_query":"impacto violencia deserción escolar abandono educación tasa estudiantes zonas conflicto"},
    {"id":"Q10", "n":3, "k":6,
     "q":"¿Qué vacíos de información, subregistros o falta de datos se identifican como obstáculo para medir la violencia real?",
     "rag_query":"vacíos información subregistro cifra negra limitaciones datos medir violencia real obstáculos"},
]

NIVELES = {1:"Extracción directa", 2:"Síntesis", 3:"Razonamiento/Límites"}
resultados = []

print("=" * 65)
print("  EVALUACIÓN COMPLETA — TUTOR ANALÍTICO HÍBRIDO")
print("  Modelo: google/gemma-2-2b-it | Vector store: ChromaDB")
print("=" * 65)

for q in BANCO:
    print(f"\n{'─'*65}")
    print(f"[{q['id']}] Nivel {q['n']} — {NIVELES[q['n']]} | k={q['k']}")
    print(f" {q['q']}")
    print("─" * 65)

    t0 = time.time()
    chunks, lat_rag = recuperar_chunks(q["rag_query"], k=q["k"])

    print(f"📡 Chunks recuperados ({len(chunks)}) — Latencia RAG: {lat_rag} ms:")
    for i, c in enumerate(chunks, 1):
        rel = round((1 - c.get("dist", 0.5)) * 100, 1)
        print(f"  [{i}] {c['fuente']} | {c.get('pag','N/A')} | {rel}%")
        print(f"      {c['texto'][:110].strip()}...")

    respuesta, _, _, lat_gen = tutor_responde_gemma(q["q"], k=q["k"], verbose=False)
    t_total = round((time.time() - t0) * 1000, 1)

    if not respuesta:
        respuesta = "La información proporcionada en el corpus no detalla este aspecto."

    print(f"\n RESPUESTA DEL TUTOR:\n{respuesta}")

    if q["id"] == "Q9":
        frases_ok = ["no detalla", "no contiene", "no se encuentra",
                     "no hay", "no aborda", "no especifica"]
        ok = any(f in respuesta.lower() for f in frases_ok)
        print(f"\n{'' if ok else ''} Anti-alucinación: {'CORRECTO' if ok else 'FALLÓ'}")

    print(f"\n  RAG: {lat_rag}ms | Gen: {lat_gen}ms | Total: {t_total}ms")
    print(f" Fuentes: {list({c['fuente'] for c in chunks})}")

    resultados.append({
        "id":          q["id"],
        "nivel":       q["n"],
        "k":           q["k"],
        "pregunta":    q["q"],
        "rag_query":   q["rag_query"],
        "respuesta":   respuesta,
        "chunks":      [c["id"] for c in chunks],
        "fuentes":     list({c["fuente"] for c in chunks}),
        "lat_rag_ms":  lat_rag,
        "lat_gen_ms":  lat_gen,
        "lat_total_ms": t_total,
    })

with open("resultados_evaluacion.jsonl", "w", encoding="utf-8") as f:
    for r in resultados:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\n{'='*65}")
print(f" Evaluación completa: {len(resultados)} preguntas")
print(" Guardado en: resultados_evaluacion.jsonl")

  EVALUACIÓN COMPLETA — TUTOR ANALÍTICO HÍBRIDO
  Modelo: google/gemma-2-2b-it | Vector store: ChromaDB

─────────────────────────────────────────────────────────────────
[Q1] Nivel 1 — Extracción directa | k=5
 ¿Cuáles son las tres entidades federativas con mayor índice de homicidios dolosos según los datos más recientes del corpus?
─────────────────────────────────────────────────────────────────
📡 Chunks recuperados (5) — Latencia RAG: 16.5 ms:
  [1] pregunta1 (1) | aprox. pág 2 | 72.9%
      tendencia cambió recientemente: Desde 2020 observamos que una de las clasificaciones aumenta (Otros delitos qu...
  [2] HALLAZGOS2023 (1) | aprox. pág 177 | 63.0%
      Homicidio 4.32% Despojo 6.34% Robo 22.52% Femenino Violencia familiar 30.08% Robo 24.49% Homicidio 7.48% Abuso...
  [3] HALLAZGOS2023 (1) | aprox. pág 166 | 60.2%
      Guana- juato y Morelos también presentan tasas significativa- mente elevadas, oscilando entre 12 y 15 muertes...
  [4] pregunta1 (1) | aprox. pág 9 | 59.8%
     

In [11]:
import time

print("=" * 65)
print("  CHAT INTERACTIVO — TUTOR ANALÍTICO HÍBRIDO")
print(f"  Modelo: google/gemma-2-2b-it | Corpus: {len(pdf_paths)} PDFs")
print("  Escribe 'salir' para terminar.")
print("=" * 65)

# Palabras que indican pregunta sobre el corpus mismo
PALABRAS_META = [
    "cuántos pdf", "cuantos pdf", "qué pdf", "que pdf",
    "cuáles pdf", "cuales pdf", "qué archivos", "que archivos",
    "cuáles archivos", "cuales archivos", "nombre", "autores",
    "autor", "cargaste", "cargados", "conforman", "corpus tiene",
    "lista de", "qué documentos", "que documentos"
]

def es_pregunta_meta(texto):
    t = texto.lower()
    return any(p in t for p in PALABRAS_META)

while True:
    consulta = input("\nTú: ").strip()

    if not consulta:
        continue

    if consulta.lower() in ["salir", "exit", "quit"]:
        print("\n Cerrando chat.")
        break

    # Preguntas sobre el corpus — respuesta directa sin RAG
    if es_pregunta_meta(consulta):
        fuentes = sorted({d["fuente"] for d in documentos_chunks})
        print(f"\n Gemma: El corpus tiene {len(pdf_paths)} PDFs con "
              f"{len(documentos_chunks)} chunks indexados en ChromaDB.\n")
        print(" Documentos cargados:")
        for f in fuentes:
            n = len([d for d in documentos_chunks if d["fuente"] == f])
            print(f"  • {f} ({n} chunks)")
        print()
        continue

    # Consulta analítica normal — pasa por RAG
    print(" Buscando en el corpus...\n")

    respuesta, chunks, lat_rag, lat_gen = tutor_responde_gemma(
        consulta, k=5, verbose=False
    )

    if not respuesta:
        respuesta = "La información proporcionada en el corpus no detalla este aspecto."

    print(f" Gemma: {respuesta}")
    print(f"\n Fuentes: {list({c['fuente'] for c in chunks})}")
    print(f" RAG: {lat_rag}ms | Gen: {lat_gen}ms")
    print("─" * 60)

  CHAT INTERACTIVO — TUTOR ANALÍTICO HÍBRIDO
  Modelo: google/gemma-2-2b-it | Corpus: 12 PDFs
  Escribe 'salir' para terminar.

 PDFs cargados (12):
  • Actor-Profile-The-Jalisco-New-Generation-Cartel-CJNG-Spanish-Translation (1) (20 chunks)
  • DesplazamientoForzado (1) (33 chunks)
  • Desplazamiento_Forzado_Interno (1) (139 chunks)
  • HALLAZGOS2023 (1) (425 chunks)
  • Medicion-multidimensional-de-la-pobreza-en-Mexico (1) (11 chunks)
  • Militarizacion (1) (32 chunks)
  • extorsiones (1) (38 chunks)
  • ong (1) (36 chunks)
  • pregunta1 (1) (13 chunks)
  • tierra_caliente_narco_autodefensa (1) (31 chunks)
  • violencia rural (1) (62 chunks)
  • violencia urbana (1) (237 chunks)


Tú: Cuantos pdfs tiene el corpus

 Gemma: El corpus tiene 12 PDFs con 1077 chunks indexados en ChromaDB.

 Documentos cargados:
  • Actor-Profile-The-Jalisco-New-Generation-Cartel-CJNG-Spanish-Translation (1) (20 chunks)
  • DesplazamientoForzado (1) (33 chunks)
  • Desplazamiento_Forzado_Interno (1) (139 c

## BLOQUE 9 — Descargar entregables

In [ ]:
from google.colab import files
files.download("dataset_finetuning.jsonl")
files.download("resultados_evaluacion.jsonl")
print(" Archivos descargados")